# Exercise 11 — The Trilemma of Land Use


## Task

You have **200 hectares** of land. Land can be allocated to wheat production, wind power, or a natural reserve. The original exercise assigns the following utility and land requirements:

| Land use | Utility per unit | Land per unit (ha) |
|---|---:|---:|
| Wheat | 11 | 3 |
| Wind farm | 9 | 3 |
| Natural reserve | 8 | 4 |

The basic integer model is

$$
\max_{x_1,x_2,x_3}\quad 11x_1+9x_2+8x_3
$$

subject to

$$
3x_1+3x_2+4x_3\le200,
$$

$$
x_1,x_2,x_3\in\mathbb N_0.
$$

First solve this integer program. Then add all of the following conditions:

- fixed costs of 400, 200, and 300 for wheat, wind farms, and natural reserves, respectively;
- if wind farms are built, they may occupy at most **90 hectares**;
- if the combined wheat and wind-farm area exceeds **50 hectares**, natural-reserve area must be at least **10 hectares**.

## Supplied reference implementation


In [1]:
using HiGHS, JuMP

land_use = ["wheat", "windFarm", "naturalReserve"]
utility = Dict(zip(land_use, [11, 9, 8]))
area = Dict(zip(land_use, [3, 3, 4]))

m = Model(HiGHS.Optimizer)
@variable(m, x[land_use] >= 0, Int)
@constraint(m, max_land, sum(x[l]*area[l] for l in land_use) <= 200)
@objective(m, Max, sum(x[l]*utility[l] for l in land_use))
optimize!(m)
round.(Int, value.(x))

# # add fixed costs
BIG = 10000
fixed_cost = Dict(zip(land_use, [400, 200, 300]))
m = Model(HiGHS.Optimizer)
@variable(m, x[land_use] >= 0, Int)
@variable(m, y[land_use], Bin)
@constraint(m, max_land, sum(x[l]*area[l] for l in land_use) <= 200)
@constraint(m, big_constraint[l=land_use], x[l] <= BIG*y[l])
@objective(m, Max, sum(x[l]*utility[l]-y[l]*fixed_cost[l] for l in land_use))
println(m)
optimize!(m)
round.(Int, value.(x))

# # add maximum wind farm constraint
@constraint(m, x["windFarm"] <= BIG*y["windFarm"])
@constraint(m, 90 - x["windFarm"]*area["windFarm"] >= -BIG*(1-y["windFarm"]))
optimize!(m)
round.(Int, value.(x))
value.(y)
objective_value(m)

# # add windfarm + wheat constraint
@variable(m, q, Bin)
@constraint(m, x["windFarm"]*area["windFarm"] + x["wheat"]*area["wheat"] - 50 <= BIG*(1-q))
@constraint(m, -(x["naturalReserve"]*area["naturalReserve"] - 10) <= BIG*q)
optimize!(m)
round.(Int, value.(x))


Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: Apple Accelerate 
MIP has 1 row; 3 cols; 3 nonzeros; 3 integer variables (0 binary)
Coefficient ranges:
  Matrix  [3e+00, 4e+00]
  Cost    [8e+00, 1e+01]
  Bound   [0e+00, 0e+00]
  RHS     [2e+02, 2e+02]
Presolving model
1 rows, 3 cols, 3 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-1); columns 0(-3); nonzeros 0(-3) - Reduced to empty
Presolve: Optimal

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Obje

1-dimensional DenseAxisArray{Int64,1,...} with index sets:
    Dimension 1, ["wheat", "windFarm", "naturalReserve"]
And data, a 3-element Vector{Int64}:
  0
  0
 50